## LangChain이란 LLM을 여러 기능과 연결해서 하나의 애플리케이션으로 만들기 쉽게 해주는 프레임 워크


In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.output_parsers import PydanticOutputParser


class CommentModeration(BaseModel):
    toxicity: Literal["safe", "warning", "toxic"] = Field(
        description="댓글의 악성 정도. safe는 안전, warning은 주의가 필요한 댓글, toxic은 악성 댓글"
    )

    contains_profanity: bool = Field(
        description="욕설이나 비속어가 포함되어 있는지 여부"
    )

    contains_personal_attack: bool = Field(
        description="특정 개인이나 대상을 직접적으로 공격하거나 비하하는지 여부"
    )

    reason: Optional[str] = Field(
        default=None, description="악성 요소가 있다면 그 이유"
    )


llm = ChatOllama(model="gemma3:4b", temperature=0.0)

parses = PydanticOutputParser(pydantic_object=CommentModeration)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 SNS에 댓글을 분석하여 악성 댓글을 자동으로 필터링 해주는 AI야. \n\n{format_instructions}",
        ),
        ("human", "댓글 : {comment}"),
    ]
).partial(format_instructions=parses.get_format_instructions())

chain = prompt | llm | parses


comments = [
    "제품이 정말 마음에 들어요. 배송도 빠르고 포장도 깔끔했습니다.",
    "이딴 것도 상품이라고 파냐? 진짜 개같네.",
    "판매자님 설명도 제대로 안 읽고 물건 보내시나요? 일을 이렇게밖에 못하세요?",
    "제품은 괜찮은데 가격이 조금 비싼 것 같아요.",
    "서비스가 너무 엉망이네요. 담당자는 진짜 무능한 것 같습니다.",
    "와 진짜 존나 맛없어요. 돈 아까워 죽겠네.",
    "상품 자체는 괜찮지만 직원 응대가 너무 불친절했습니다.",
    "판매자 너는 장사할 자격이 없다. 머리가 있으면 이런 식으로 운영하지 마라.",
]

for comment in comments:
    res = chain.invoke({"comment": comment})
    print(f"'{comment}")
    print(f"toxicity : {res.toxicity}")
    print(f"욕설 포함 여부 : {res.contains_profanity}")
    print(f"인신 공격 여부 : {res.contains_personal_attack}")
    if res.reason:
        print(f"문제가 있을 때 그 이유 : {res.reason}")
    print()

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.output_parsers import PydanticOutputParser


class CommentModeration(BaseModel):
    toxicity: Literal["safe", "warning", "toxic"] = Field(
        description="댓글의 악성 정도. safe는 안전, warning은 주의가 필요한 댓글, toxic은 악성 댓글"
    )

    contains_profanity: bool = Field(
        description="욕설이나 비속어가 포함되어 있는지 여부"
    )

    contains_personal_attack: bool = Field(
        description="특정 개인이나 대상을 직접적으로 공격하거나 비하하는지 여부"
    )

    reason: Optional[str] = Field(
        default=None, description="악성 요소가 있다면 그 이유"
    )


llm = ChatOllama(model="gemma3:4b", temperature=0.0)

parses = PydanticOutputParser(pydantic_object=CommentModeration)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 SNS에 댓글을 분석하여 악성 댓글을 자동으로 필터링 해주는 AI야. \n\n{format_instructions}",
        ),
        ("human", "댓글 : {comment}"),
    ]
).partial(format_instructions=parses.get_format_instructions())

chain = prompt | llm | parses


comments = [
    "제품이 정말 마음에 들어요. 배송도 빠르고 포장도 깔끔했습니다.",
    "이딴 것도 상품이라고 파냐? 진짜 개같네.",
    "판매자님 설명도 제대로 안 읽고 물건 보내시나요? 일을 이렇게밖에 못하세요?",
    "제품은 괜찮은데 가격이 조금 비싼 것 같아요.",
    "서비스가 너무 엉망이네요. 담당자는 진짜 무능한 것 같습니다.",
    "와 진짜 존나 맛없어요. 돈 아까워 죽겠네.",
    "상품 자체는 괜찮지만 직원 응대가 너무 불친절했습니다.",
    "판매자 너는 장사할 자격이 없다. 머리가 있으면 이런 식으로 운영하지 마라.",
]

for comment in comments:
    res = chain.invoke({"comment": comment})
    print(f"'{comment}")
    print(f"toxicity : {res.toxicity}")
    print(f"욕설 포함 여부 : {res.contains_profanity}")
    print(f"인신 공격 여부 : {res.contains_personal_attack}")
    if res.reason:
        print(f"문제가 있을 때 그 이유 : {res.reason}")
    print()

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.output_parsers import PydanticOutputParser


# 스키마 — 카테고리 + 개선 제안 추가
class ReviewFull(BaseModel):
    rating: int = Field(description="평점 1-5점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정")
    category: Literal["품질", "가격", "배송", "디자인", "기타"] = Field(
        description="리뷰가 다루는 영역"
    )
    keywords: list[str] = Field(description="핵심 키워드 3개")
    improvement: Optional[str] = Field(
        default=None, description="개선 제안 (있으면, 없으면 None)"
    )


llm = ChatOllama(model="gemma3:4b", temperature=0.7)


parser_r = PydanticOutputParser(pydantic_object=ReviewFull)
prompt_r = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "쇼핑몰 리뷰 분석가입니다. 한국어로 답하세요.\n\n{format_instructions}",
        ),
        ("human", "리뷰: {review}"),
    ]
).partial(format_instructions=parser_r.get_format_instructions())

chain_review = prompt_r | llm | parser_r


# 테스트 — 3개 리뷰
reviews = [
    "포장이 너무 아쉽네요. 박스가 찌그러져 왔어요.",
    "가성비 짱! 이 가격에 이 정도면 최고예요.",
    "디자인은 예쁜데 기능이 좀 부족하네요.",
]

for r in reviews:
    res = chain_review.invoke({"review": r})
    print(f"📝 '{r}'")
    print(f"   ⭐ {res.rating}/5 | {res.sentiment} | 📁 {res.category}")
    print(f"   🏷️  {', '.join(res.keywords)}")
    if res.improvement:
        print(f"   💡 개선: {res.improvement}")
    print()

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}


llm = ChatOllama(model="gemma3:4b", temperature=0.7)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {persona}입니다."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)


def get_session_id_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain = RunnableWithMessageHistory(
    prompt | llm,
    get_session_id_history,
    input_messages_key="question",
    history_messages_key="history",
)


result = chain.invoke(
    {
        "question": "AI 엔지니어 직무를 꿈꾸는 신입들이 뭘 준비하면 좋을까 핵심역량으로",
        "persona": "AI 엔지니어를 10년이상 직무를 수행한 전문가",
    },
    config={"configurable": {"session_id": "test_01"}},
)

print(result.content)

In [1]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

python-dotenv could not parse statement starting at line 30


True

In [2]:
llm = ChatNVIDIA(model=os.getenv("NVIDIA_MODEL"), api_key=os.getenv("NVIDIA"))

prompt = ChatPromptTemplate.from_messages(
    [("system", "당신은 {persona} 입니다."), ("human", "{question}")]
)

chain = prompt | llm

response = chain.invoke(
    {"question": "이상치란 무엇인지 한 문장으로.", "persona": "데이터 분석가"}
)

In [3]:
print(response.content)

이상치는 평균이나 기대값에서 크게 벗어나 통계적 변동 범위 밖에 있는 데이터 포인트를 말합니다.


In [ ]:
PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}

question = "좋은 데이터란 무엇인가요?"

for name in PERSONAS.keys():
    result = chain.invoke({"persona": PERSONAS[name], "question": question})
    print("[", name, "]")
    print(result.content)
    print()

In [21]:
def ask(persona_name: str, question: str) -> str:
    system_message = PERSONAS[persona_name]
    for chunk in chain.stream({"persona": system_message, "question": question}):
        print(chunk.content, end="", flush=True)

In [22]:
answer = ask(
    "분석가", "promptTemplate과 ChatPromptTemplate의 차이점을 한줄로 설명해주세요."
)
print(answer)

promptTemplate은 **한 개의 텍스트 입력(1개 메시지)**에만 사용되는 반면, ChatPromptTemplate은 **2개 이상의(시스템·사용자·어시스턴트 등) 메시지를 조합해 멀티턴 대화를 구성**하도록 설계됩니다(예: 최소 2 ~ 4 메시지).None


In [16]:
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("종료합니다.")
        return current_persona, True

    if cmd == "/persona":
        print("현재 페르소나 : ", current_persona)
    elif cmd == "/help":
        print("명령어 : /quit, /exit, /chant<이름>, /persona, /help")
    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False  # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

In [17]:
current = "분석가"
current, _ = handle_command("/persona", current)
current, _ = handle_command("/change 마케터", current)
current, _ = handle_command("/persona", current)
current, _ = handle_command("/change 외계인", current)  # 잘못된 페르소나
current, _ = handle_command("/help", current)

현재 페르소나 :  분석가
✅ 페르소나가 '마케터'(으)로 변경되었습니다.
현재 페르소나 :  마케터
❌ '외계인'은 등록된 페르소나가 아닙니다.
   사용 가능: 분석가, 마케터, 개발자, 기획자, 디자이너
명령어 : /quit, /exit, /chant<이름>, /persona, /help


In [26]:
from datetime import datetime


def create_log(log: list, persona: str, input: str, output: str):
    log.append(
        f"[{datetime.now().strftime('%H:%M:%S')}] [{persona}] {input} -> {output}"
    )


In [27]:
def run_chatbot(initial_persona: str = "분석가"):
    log = []
    persona = initial_persona

    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("<질문>").strip()
        if not user_input:
            continue

        if user_input.startswith("/"):
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:
            answer = ask(persona, user_input)
            create_log(log, persona, user_input, answer)

            print(f"[{persona}] {answer}\n")
            print()
            print("===" * 25)

    with open("chat_log.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(log))

In [28]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

데이터 분석가는 대규모 데이터를 정제·시각화·통계·머신러닝으로 인사이트를 도출해 조직의 의사결정에 가치를 제공하는 역할이며, 2026년 기준 미국에서 평균 연봉이 약 $120 k, 채용지수 3% 이하, 매년 15 % 성장률로 기록되고 있습니다.[분석가] None


안녕히 가세요![분석가] None


종료합니다.


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(model=os.getenv("NVIDIA_MODEL"), api_key=os.getenv("NVIDIA"))

print(llm.invoke("내 이름은 박찬룡이야."))
print(llm.invoke("내 이름이 뭐라고?"))

In [33]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

history = [SystemMessage("당신은 AI 어시스턴트입니다.")]

history.append(HumanMessage("내 이름은 박찬룡이야"))
result = llm.invoke(history)
history.append(AIMessage(result.content))
print(result.content)


history.append(HumanMessage("내 이름이 뭐였지?"))
result = llm.invoke(history)
history.append(AIMessage(result.content))
print(result.content)

안녕하세요, 박찬룡님! 오늘 무엇을 도와드릴까요? 필요하신 정보나 궁금한 점이 있으시면 언제든 말씀해 주세요.
네, 박찬룡님이시죠!💬 찾으시는 내용이나 궁금한 점이 있으시면 언제든 말씀해주세요.


In [35]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트입니다. 한국어로 답하세요."),
        MessagesPlaceholder("history"),  # ← 이전 대화 메시지들이 자동으로 여기에 들어감
        ("human", "{question}"),  # ← 새 사용자 입력
    ]
)

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    print("store[session_id] : ", store[session_id])
    return store[session_id]

In [38]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain = prompt | llm

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user_01"}}

result = chain_with_memory.invoke({"question": "내 이름은 찬룡이야"}, config=config)

print(result.content)

print()

result2 = chain_with_memory.invoke(
    {"question": "내 이름이 뭔 지 말해봐"}, config=config
)

print(result2.content)

/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


store[session_id] :  
네, 찬룡님! 반갑습니다. 무엇을 도와드릴까요?

store[session_id] :  Human: 내 이름은 찬룡이야
AI: 네, 찬룡님! 반갑습니다. 무엇을 도와드릴까요?
귀하의 이름은 “찬룡”입니다.


In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

llm = ChatNVIDIA(model=os.getenv("NVIDIA_MODEL"), api_key=os.getenv("NVIDIA"))

store = {}

prompt = ChatPromptTemplate(
    [
        ("system", "당신은 {persona} 입니다."),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)


chain = prompt | llm

PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}


def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]


chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)


def ask(persona_name: str, question: str) -> str:
    system_message = PERSONAS[persona_name]
    for chunk in chain.stream(
        {"persona": system_message, "question": question},
        config={"configurable": {"session_id": "user_01"}},
    ):
        print(chunk.content, end="", flush=True)


def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("종료합니다.")
        return current_persona, True

    if cmd == "/persona":
        print("현재 페르소나 : ", current_persona)
    elif cmd == "/help":
        print("명령어 : /quit, /exit, /chant<이름>, /persona, /help")
    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False  # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False


def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona

    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("<질문>").strip()
        if not user_input:
            continue

        if user_input.startswith("/"):
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")
            print()
            print("===" * 25)

/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [41]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

안녕하세요 찬룡님!  
10년 경력 분석가로서, 데이터 수집·전처리부터 시각화·전략까지 매 10개 이상의 프로젝트를 성공적으로 이끌었으며, 평균 프로젝트 기간은 4주 내에 완료합니다. 언제든 분석 관련 고민이나 데이터 개선 아이디어를 말씀해 주세요![분석가] None


찬룡입니다.[분석가] None


저는 **ChatGPT(Version 4)**입니다.  
- 개발사: OpenAI  
- 출시일: 2023년 11월  
- 학습 데이터 시점: 2023년 말까지(2024 년 초까지)  

이상입니다.[분석가] None


알겠습니다. 필요하시면 언제든 돌아와 주세요![분석가] None


종료합니다.


In [45]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate

llm = ChatNVIDIA(model=os.getenv("NVIDIA_MODEL"), api_key=os.getenv("NVIDIA"))

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 리뷰 분석가입니다. 한국어로 답하세요."),
        ("human", "이 리뷰를 분석해주세요. {review}"),
    ]
)

chain = prompt | llm

for chunk in chain.stream(
    {"review": "이 카메라 진짜 좋아요! 가격도 싸고 화질도 최고!"}
):
    print(chunk.content, end="", flush=True)

### 리뷰 분석 보고서  

| 항목 | 내용 |
|------|------|
| **전반적인 감정** | **매우 긍정적** — ‘진짜 좋아요’라는 표현은 강한 만족도를 나타냅니다. |
| **핵심 포인트** | 1. **가격**: “싸고”라는 단어는 가성비 뛰어남을 강조합니다. <br>2. **화질**: “최고”라는 표현으로 영상·사진 품질이 만족스럽다는 의미. |
| **추가적으로 유추되는 장점** | • **편리함**: 가격이 싸면서 품질이 좋다면 사용이 간편·저렴한 투자 효과가 큼.<br>• **대상층**: 예산이 한정적이면서도 사진·영상에 취미나 사무용으로 관심이 있는 사람에게 적합. |
| **잠재적인 단점(언급되지 않음)** | • **배터리 수명**<br>• **내구성** (테두리, 방수 등)<br>• **추가 기능** (웨어러블, Wi‑Fi 등) |
| **정성적 해석** | 리뷰어는 **가격 대비 품질**을 가장 핵심 가치로 삼고 있습니다. 가격이 저렴하면서 화질이 우수하면 ‘가성비가 매우 좋다’는 인식이 강하게 전달됩니다. 두 문장만으로도 감정적 열정(“진짜 좋아요”)과 객관적 매출 포인트(가격·화질)를 동시에 제시해 상담이나 마케팅에 유용합니다. |
| **마케팅 활용** | 1. **키워드**: “가성비 최고”, “저렴한 가격”, “높은 화질” <br>2. **프로모션**: “가격 대비 최고 화질”을 강조한 할인·리뷰 캠페인에 활용. <br>3. **대상**: 예산이 한정된 학생, 여행가, 일상 기록자. |
| **향후 개선 및 리서치 제안** | • 실질적인 사용 만족도(배터리, 조작성 등)에 대한 추가 리뷰 수집.<br>• ‘가성비’에 대한 객관적 비교(시중 다른 카메라와 성능/가격) 분석.<br>• 리뷰어와의 인터뷰를 통해 화질에 만족한 구체적 상황(조명, 거리 등) 파악. |

> **결론**  
> 이 리뷰는 **가격이 저렴하면서 화질이 탁월**한 카메라의 가치를 강하게 어필하고 있습니다. 짧은 문장 안에서도 감성(좋아요)과

In [47]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

for chunk in chain.stream(
    {"review": "이 카메라 진짜 좋아요! 가격도 싸고 화질도 최고!"}
):
    print(chunk, end="", flush=True)

### 리뷰 요약  
- **긍정적 핵심 내용**  
  - “**카메라가 진짜 좋아요**” → 사용자가 제품에 전반적으로 만족하고 있다는 신호.  
  - “**가격도 싸고**” → 가격 대비 가치가 높다는 것을 강조.  
  - “**화질도 최고**” → 사진/비디오 품질이 뛰어나다는 중요 포인트.  

### 감정 및 톤  
- **긍정적, 설득력 있는 어조**  
  - 단호하면서도 간결한 문장 구조로 신뢰감을 전달.  
  - 감정 표현은 “좋아요”와 “최고”라는 강한 긍정어를 사용해 흥정함.  

### 핵심 매점  
1. **가격 경쟁력** – ‘싸다’라는 표현으로 가성비가 뛰어남을 부각.  
2. **의도적 명확성** – 화질을 “최고”라 고백함으로써 성능에 대한 확신을 줌.  
3. **정성적 만족** – 사용자 경험이 실제로 긍정적이었음을 간접적으로 증언.  

### 사용제안  
- **대상 고객**: 예산이 제한적이면서도 사진·비디오 품질을 중요시하는 소비자(블로거, 소셜미디어 인플루언서).  
- **마케팅 포인트**: “가성비 최고 카메라” 혹은 “가격 대비 최고의 화질 제공” 등으로 강조하면 좋음.  
- **추가 정보 활용**: 실제 사진샘플과 비교 사진을 함께 제공하면 신뢰도를 더욱 높일 수 있음.  

### 개선 방향  
- 현재 리뷰는 아주 짧아 “세부 사용 환경”(예: 야간 촬영 성능, 배터리 수명 등)이나 “단점”이 없어서 부정적 즉 선택사항이 없습니다.  
- 만약 시장 경쟁이 치열하다면, 추가적인 예시(다른 모델과 비교)나 **실제 사용 후기**를 더 구비해 데이터 기반 신뢰도 상승을 도모할 수 있습니다.  

> **결론**  
> 이 리뷰는 가격 대비 성능, 특히 화질에 중점을 두고 있으며, 긍정적인 소비자 경험을 강하게 어필합니다. 마케팅에서는 가성비와 고화질을 핵심 키워드로 삼아 타깃 고객에게 어필하는 것이 효과적일 것입니다.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional


class Person(BaseModel):
    name: str
    age: int


p1 = Person(name="홍길동", age=40)
print(p1.name, p1.age)


홍길동 40


In [49]:
from langchain_core.output_parsers import PydanticOutputParser


class ReviewAnalysis(BaseModel):
    rating: int = Field(description="평점 1-5점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정")
    keywords: list[str] = Field(description="핵심 키워드 3개")
    summary: str = Field(description="한 문장 요약")


parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 리뷰 분석가입니다. 모든 텍스트는 한국어로 작성하세요.\n\n"
            "{format_instructions}",
        ),
        ("human", "리뷰: {review}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
review = "이 카메라 진짜 좋아요! 가격도 싸고 화질도 최고!"
result = chain.invoke({"review": review})

print(f"타입:     {type(result).__name__}")
print(f"⭐ 평점:  {result.rating}/5")
print(f"😊 감정:  {result.sentiment}")
print(f"🏷️  키워드: {', '.join(result.keywords)}")
print(f"📝 요약:  {result.summary}")
print()

타입:     ReviewAnalysis
⭐ 평점:  5/5
😊 감정:  positive
🏷️  키워드: 카메라, 가격, 화질
📝 요약:  가격이 저렴하고 화질이 뛰어난 카메라를 칭찬하는 리뷰



In [55]:
import time


class ReviewFull(BaseModel):
    rating: int = Field(description="평점 1-5점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정")
    category: Literal["품질", "가격", "배송", "디자인", "기타"] = Field(
        description="리뷰가 다루는 영역"
    )
    keywords: list[str] = Field(description="핵심 키워드 3개")
    improvement: Optional[str] = Field(
        default=None, description="개선 제안 (있으면, 없으면 None)"
    )


parser = PydanticOutputParser(pydantic_object=ReviewFull)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "쇼핑몰 리뷰 분석가입니다. 한국어로 답하세요.\n\n{format_instructions}",
        ),
        ("human", "리뷰: {review}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

reviews = [
    "포장이 너무 아쉽네요. 박스가 찌그러져 왔어요.",
    "가성비 짱! 이 가격에 이 정도면 최고예요.",
    "디자인은 예쁜데 기능이 좀 부족하네요.",
]

start = time.time()

# result_batch = chain.batch(reviews)

for review in reviews:
    result = chain.invoke({"review": review})

    print(f"📝 '{review}'")
    print(f"   ⭐ {result.rating}/5 | {result.sentiment} | 📁 {result.category}")
    print(f"   🏷️  {', '.join(result.keywords)}")
    if result.improvement:
        print(f"   💡 개선: {result.improvement}")
    print()

time_batch = time.time() - start

print(f"⚡ batch:    {time_batch:.2f}초")

📝 '포장이 너무 아쉽네요. 박스가 찌그러져 왔어요.'
   ⭐ 2/5 | negative | 📁 배송
   🏷️  포장, 박스, 찌그러짐
   💡 개선: 튼튼한 포장재 사용

📝 '가성비 짱! 이 가격에 이 정도면 최고예요.'
   ⭐ 5/5 | positive | 📁 가격
   🏷️  가성비, 가격, 최고

📝 '디자인은 예쁜데 기능이 좀 부족하네요.'
   ⭐ 2/5 | negative | 📁 디자인
   🏷️  예쁜, 기능 부족, 디자인
   💡 개선: 기능을 보강해 사용자 편의성을 높여 주세요.

⚡ batch:    39.73초


In [59]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


class ReviewAnalysis(BaseModel):
    rating: int = Field(description="평점 1-5점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정")
    keywords: list[str] = Field(description="핵심 키워드 3개")


parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)

llm = ChatNVIDIA(model=os.getenv("NVIDIA_MODEL"), api_key=os.getenv("NVIDIA"))

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "리뷰 분석가입니다. 한국어로 답하세요.\n\n{format_instructions}"),
        ("human", "리뷰: {review}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

prompt_r = ChatPromptTemplate(
    [
        (
            "system",
            "쇼핑몰 고객센터 응대 담당자입니다. "
            "리뷰에 진심을 담아 한국어로 3-4문장의 답장을 작성하세요. "
            "긍정 리뷰엔 감사를, 부정 리뷰엔 사과와 개선 의지를 표현하세요.",
        ),
        ("human", "고객 리뷰: {review}"),
    ]
)

chain = prompt | llm | parser

chain_reply = prompt_r | llm | StrOutputParser()

review = "이 카메라 진짜 좋아요!"

start = time.time()
analysis = chain.invoke({"review": review})
reply = chain_reply.invoke({"review": review})


time = time.time() - start

print(time)

13.687795162200928


In [63]:
import time
from langchain_core.runnables import RunnableParallel

full_chain = RunnableParallel(anl=chain, reply=chain_reply)

start = time.time()
result = full_chain.invoke({"review": review})
parallel_time = time.time() - start

print(f"⏱️  병렬 호출: {parallel_time:.2f}초")

⏱️  병렬 호출: 13.46초


In [65]:
from langchain_core.runnables import RunnableLambda


def add_alert(analysis):
    if analysis.rating <= 2:
        return {"level": "🚨 긴급 응대", "data": analysis}
    elif analysis.rating <= 3:
        return {"level": "⚠️ 주의 필요", "data": analysis}
    else:
        return {"level": "🟢 정상", "data": analysis}


alert_step = RunnableLambda(add_alert)

new_chain = chain | alert_step

for review in ["완전 만족!", "그냥 그래요", "최악이에요. 환불해주세요!"]:
    r = new_chain.invoke({"review": review})
    print(f"{r['level']:15s} | {review}")

🟢 정상            | 완전 만족!
⚠️ 주의 필요        | 그냥 그래요
🚨 긴급 응대         | 최악이에요. 환불해주세요!


In [68]:
from langchain_core.runnables import RunnablePassthrough

full_chain = RunnableParallel(al=chain, ply=chain_reply, origin=RunnablePassthrough())

result = full_chain.invoke({"review": "포장이 너무 아쉬워요."})
print(f"📝 원본 입력: {result['origin']['review']}")
print(f"⭐ 평점: {result['al'].rating}")
print(f"💬 답장: {result['ply'][:60]}...")

📝 원본 입력: 포장이 너무 아쉬워요.
⭐ 평점: 2
💬 답장: 먼저, 포장에 만족스럽지 못한 점 진심으로 사과드립니다.  
고객님의 소중한 의견을 바로 현장에 전달해 포장...


In [ ]:
temp_list = ["A", "B", "C"]
temp_dict = {}

for idx, val in enumerate(temp_list):
    temp_dict[idx] = val

print(temp_dict)

# temp_dict[idx] = val for idx, val in enumerate(temp_list)

temp_dict = {idx: val for idx, val in enumerate(temp_list)}
print(temp_dict)

{0: 'A', 1: 'B', 2: 'C'}
